# 01 — ZITboost 단독 (경로 A)

**최종 예측 공식**: `E[Y] = (1 - π) × μ` — ZITboost 내부에서 EM으로 π·μ·φ 동시 학습.

- 모델: `zitboost` (고정)
- target_transform: `none` (Tweedie가 분포 직접 모델링, 변경 불가)
- die-level 학습 + unit-level RMSE objective (mean 집계)
- 후처리는 본 노트북에서 미사용 (`04_blend.ipynb`에서 최종 단계만)
- 출력: `{OUT_DIR}/fold_models.pkl` + `best_params.json` + die/unit 예측 CSV 6종

**경로 B(`03_zit_plus_reg.ipynb`)가 본 노트북의 `oof_die.csv / val_die.csv / test_die.csv`의 `one_minus_pi` 컬럼을 재사용**하므로 반드시 먼저 실행한다.

> ⚠ **본 노트북은 로컬 전용** — ZITboost EM은 학습 시간이 길어 Colab 세션 timeout 위험.

## 1. 환경 설정 + 모듈 import (로컬 전용)

In [1]:
import os, sys

# ── 로컬 전용 setup ──
%run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs
from utils.evaluate import rmse

MODEL_ROOT = os.path.join(PROJECT_ROOT, "3_modeling")
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess, hpo, models

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Available models: {models.AVAILABLE_MODELS}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
Available models: ['lgbm', 'xgb', 'catboost', 'et', 'enet', 'zitboost']


## 2. 실험 설정

- `EXP_ID` / `EXP_MEMO` / `USER` : 실험 식별
- `N_TRIALS` / `N_FOLDS` : Optuna 예산
- `PARAMS` : 전처리 파라미터 override (빈 dict면 `preprocess.DEFAULT_PARAMS` 그대로)
- `TARGET_TRANSFORM` : ZITboost는 **`'none'` 고정** (바꿀 수 없음)

In [2]:
# ── 실험 식별 ──
EXP_ID   = 'zit-final-002'
EXP_MEMO = 'ZITboost 단독, 전처리 고정'
USER     = 'jh'

# ── Optuna 예산 ──
N_TRIALS = 1
N_FOLDS  = 5

# ── REUSE 모드 (기존 best_params.json 재사용 → HPO 스킵, refit만 실행) ──
# 전처리만 바꾸고 HP는 그대로 써서 "개선 여부 저비용 검증" 용도.
REUSE_BEST_PARAMS     = False   # ★ True: 기존 HP 재사용 / False: 정상 HPO
PREV_BEST_PARAMS_PATH = ''      # REUSE 시 기존 best_params.json 경로 (로컬)

# ── 타깃 처리 (ZIT는 고정) ──
TARGET_TRANSFORM = 'none'   # ★ ZIT 고정, 변경 불가
CLIP_Y_EXTREME   = True

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, 'final', 'zit_only')
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 전처리 파라미터 override ──
# target_transform='none' (ZIT 내부 Tweedie 가 직접 모델링하므로 항상 none)
# 1차 Optuna Two-Stage DB (43 study, 3,460 none-trial) top-50 mode 기반.
# plan §9.5 L4: corr_keep_by='target_corr' 는 fold leakage 이므로 'std' 로 강제.
PARAMS = {
    'missing_threshold':          0.4,       # none top-50 mode
    'corr_threshold':             0.90,      # none top-50 mode
    'corr_keep_by':               'std',     # plan 금기 준수
    'add_indicator':              True,      # none top-50 mode
    'indicator_threshold':        0.05,      # none top-50 mode
    'spatial_max_dist':           5.0,       # none top-50 mode
    'post_impute_corr_threshold': 0.98,      # none top-50 mode
    'post_impute_corr_keep_by':   'std',     # 전 transform 공통 best
}

# ── 디바이스 ──
# models.DEVICE = 'gpu'   # GPU 사용 시 주석 해제

print(f'EXP: {EXP_ID} | USER: {USER} | model: zitboost (고정)')
print(f'N_TRIALS={N_TRIALS}, N_FOLDS={N_FOLDS}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} (고정) | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'REUSE_BEST_PARAMS={REUSE_BEST_PARAMS}'
      + (f' (path={PREV_BEST_PARAMS_PATH})' if REUSE_BEST_PARAMS else ''))
print(f'OUT_DIR={OUT_DIR}')
print(f'DEVICE={models.DEVICE}')

EXP: zit-final-002 | USER: jh | model: zitboost (고정)
N_TRIALS=1, N_FOLDS=5
TARGET_TRANSFORM=none (고정) | CLIP_Y_EXTREME=True
REUSE_BEST_PARAMS=False
OUT_DIR=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\zit_only
DEVICE=cpu


## 3. 데이터 로드 + target clip + 전처리 실행

`preprocess.run()`이 cleaning(11 파라미터) + outlier winsorize(0.0, 0.99 고정)를 수행. 매번 실행 (캐시 없음).

In [3]:
# ── 데이터 로드 ──
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')
print(f'ys keys: {list(ys.keys())}')

# ── y_train 극단값 clip (1.0 → second_max) ──
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

# ── 전처리 (고정 파이프라인, PARAMS로 override) ──
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

print(f'\n[전처리 완료]')
print(f'  train: {xs_train.shape}')
print(f'  val:   {xs_val.shape}')
print(f'  test:  {xs_test.shape}')
print(f'  feat_cols: {len(feat_cols_clean)}')
print(f'  effective_params: {pp["effective_params"]}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
xs: (174572, 1091), feat_cols: 1087
ys keys: ['train', 'validation', 'test', 'all']
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개 샘플
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1034 (53개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1034
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 929개
    컬럼: 1034 → 929 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=40%
  제거: 5개, 잔여: 924개
    컬럼: 929 → 924 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 897개
    컬럼: 924 → 897 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 333개, 잔여: 564개
    컬럼: 897 → 564 (333개 제거)
    DataFrame: (104748, 621)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imp

## 4. Optuna HPO — zitboost

In [4]:
import json

best_params_for_refit = None
already_resolved = False
study = None

# 재현성 메타 (REUSE 여부와 무관하게 채움)
study_meta_for_save = {
    'exp_id':            EXP_ID,
    'exp_memo':          EXP_MEMO,
    'user':              USER,
    'model_name':        'zitboost',
    'target_transform':  TARGET_TRANSFORM,
    'clip_y_extreme':    CLIP_Y_EXTREME,
    'effective_pp_params': pp['effective_params'],
    'n_trials':          N_TRIALS,
    'n_folds':           N_FOLDS,
    'seed':              SEED,
    'reuse_best_params': REUSE_BEST_PARAMS,
}

if REUSE_BEST_PARAMS:
    assert PREV_BEST_PARAMS_PATH and os.path.exists(PREV_BEST_PARAMS_PATH), \
        f'PREV_BEST_PARAMS_PATH 파일 없음: {PREV_BEST_PARAMS_PATH}'
    with open(PREV_BEST_PARAMS_PATH, encoding='utf-8') as f:
        _meta = json.load(f)
    best_params_for_refit = _meta['best_params_resolved']
    already_resolved = True
    study_meta_for_save['prev_best_params_path'] = PREV_BEST_PARAMS_PATH
    print(f'[REUSE] HPO 스킵, best_params 로드: {PREV_BEST_PARAMS_PATH}')
    print(f'  model_name(prev) = {_meta.get("model_name")}')
    print(f'  keys: {list(best_params_for_refit)[:8]} ...')
else:
    res = hpo.run_hpo(
        xs_train=xs_train,
        ys_train_unit=ys_input['train'],
        feat_cols=feat_cols_clean,
        model_name='zitboost',
        n_trials=N_TRIALS,
        n_folds=N_FOLDS,
        study_name=EXP_ID,
        storage=f'sqlite:///{DB_PATH}',
        user_attrs=study_meta_for_save,
    )
    study                 = res['study']
    best_params_for_refit = res['best_params']
    study_meta_for_save['hpo_best_value'] = float(res['best_value'])
    print(f'\n[HPO 완료] best OOF RMSE = {res["best_value"]:.6f}')
    print(f'best_params = {best_params_for_refit}')

[I 2026-04-21 14:28:42,787] A new study created in RDB with name: zit-final-002


  0%|          | 0/1 [00:00<?, ?it/s]

[W 2026-04-21 14:31:12,716] Trial 0 failed with parameters: {'zeta': 1.39963209507789, 'n_em_iters': 20, 'mu_n_estimators': 1491, 'mu_learning_rate': 0.030049873591901578, 'mu_num_leaves': 67, 'mu_max_depth': 6, 'mu_min_child_samples': 10, 'mu_subsample': 0.9330880728874675, 'mu_colsample_bytree': 0.7207805082202461, 'mu_reg_alpha': 0.023585940584142682, 'mu_reg_lambda': 1.5320059381854043e-08, 'pi_n_estimators': 487, 'pi_learning_rate': 0.06798962421591129, 'pi_num_leaves': 39, 'pi_max_depth': 4, 'pi_min_child_samples': 26, 'phi_n_estimators': 187, 'phi_learning_rate': 0.03347776308515933, 'phi_num_leaves': 64, 'phi_max_depth': 4, 'phi_min_child_samples': 65} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Dell5371\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "c:\Users\Dell5371\Desktop\기업연계프로젝트\3_modeling\final\modules\hpo.py", line 236, in objective
   

KeyboardInterrupt: 

## 5. Best trial 재학습 (K-fold OOF)

In [ ]:
final = hpo.refit_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name='zitboost',
    best_params=best_params_for_refit,
    already_resolved=already_resolved,
    n_folds=N_FOLDS,
)

# train OOF 기준 unit RMSE
y_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
oof_u  = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_true.index]
oof_rmse = float(np.sqrt(np.mean((oof_u.values - y_true.values)**2)))
print(f'\n[Refit 완료] OOF unit RMSE = {oof_rmse:.6f}')
print(f'fold_models: {len(final["fold_models"])}개')
print(f'val_pred_die shape:  {final["val_pred_die"].shape}')
print(f'test_pred_die shape: {final["test_pred_die"].shape}')
print(f'π (one_minus_pi 저장 대상): oof {final["oof_pi"].shape}, '
      f'val {final["val_pi"].shape}, test {final["test_pi"].shape}')

## 6. 아티팩트 저장

- `fold_models.pkl` : 5-fold 학습된 모델 리스트
- `best_params.json` : 모델 이름 + HP
- `oof_die.csv / val_die.csv / test_die.csv` : die-level 예측 + π + μ + **one_minus_pi** (경로 B에서 재사용)
- `oof_unit.csv / val_unit.csv / test_unit.csv` : unit-level (mean 집계)

In [ ]:
# ── Postprocess 설정 (집계 6종 + π threshold + zero_clip 튜닝) ──
POSTPROCESS_CONFIG = {
    'agg_methods':         ('mean', 'median', 'max', 'min', 'trimmed_mean', 'weighted'),
    'pi_threshold_range':  (0.5, 0.95),
    'pi_threshold_step':   0.01,
    'zero_clip_range':     (0.001, 0.015),
    'zero_clip_step':      0.001,
    'use_pi_threshold':    True,   # ZITboost 는 π 활용
}

hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    postprocess_config=POSTPROCESS_CONFIG,
    study_meta=study_meta_for_save,
)

# 저장된 파일 리스트
for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:30s}  {size_kb:10,.1f} KB')

## 7. 요약

In [ ]:
print(f'★ 경로 A (ZITboost 단독)')
print(f'  EXP_ID       : {EXP_ID}')
print(f'  모델         : zitboost')
print(f'  REUSE 모드   : {REUSE_BEST_PARAMS}')
print(f'  OOF RMSE     : {oof_rmse:.6f}  (train unit, 최적화 기준)')
if not REUSE_BEST_PARAMS and study is not None:
    print(f'  HPO best val : {res["best_value"]:.6f}')
    print(f'  trials       : {len(study.trials)}')
print(f'  fold 수      : {N_FOLDS}')
print(f'  feat_cols    : {len(feat_cols_clean)}')
print(f'  저장 경로    : {OUT_DIR}')
print(f'\n다음 단계:')
print(f'  → 02_reg_only.ipynb (경로 C)')
print(f'  → 03_zit_plus_reg.ipynb (경로 B, 본 결과의 one_minus_pi 재사용)')